<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN-NEW/blob/main/07_rag/question_answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question Answering and the Road to RAG

The previous session built retrievers that find the right document. This notebook builds the next piece: readers, the models that turn a document plus a question into an answer. We run an extractive reader that copies a span out of the text and reports how confident it is, watch it break on Helcon's whole knowledge base because of the 512-token limit, then try a generative reader that can rephrase and answer yes/no questions. Both failures point at the same fix, retrieval plus reading, which is the Helcon Support Copilot we assemble in the next session.

This notebook runs on CPU. In class it is shown on screen, and everything below also stands alone for self-study.

In [ ]:
# Setup, run this cell first.
# It installs the libraries and pre-downloads the two readers of the session (~500 MB),
# so every later cell loads from cache instantly.
!pip install -q transformers torch sentencepiece requests beautifulsoup4 googlesearch-python

from huggingface_hub import snapshot_download
for repo in ["distilbert-base-uncased-distilled-squad", "t5-small"]:
    snapshot_download(repo)
    print(f"cached: {repo}")

# The course repo carries Helcon's help center, the documents our running support copilot reads.
# Locally the repo is already on disk; on Colab we clone it.
import os, sys
REPO = "https://github.com/acastellanos-ie/NLP-MBDS-EN-NEW"
if not os.path.exists("NLP-MBDS-EN") and not os.path.exists("../system/system_io.py"):
    !git clone --depth 1 -q $REPO NLP-MBDS-EN
for candidate in ("../system", "NLP-MBDS-EN/system", "system"):
    if os.path.exists(os.path.join(candidate, "system_io.py")):
        sys.path.insert(0, candidate)
        break
from system_io import load_seed
print("Setup complete.")

## Step 1: Extractive QA, a pointer with a probability

An extractive reader does not write anything. Given a question and a context paragraph, it predicts two token positions, where the answer starts and where it ends, and copies that span verbatim. We use DistilBERT fine-tuned on SQuAD: an encoder small enough for CPU that still handles fact-lookup questions over a help-center article.

The start and end scores are logits. Push them through a softmax and the model also tells us how sure it is. That probability is the only safety mechanism extractive QA has.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import DistilBertTokenizer, DistilBertForQuestionAnswering

qa_model_id = "distilbert-base-uncased-distilled-squad"
qa_tokenizer = DistilBertTokenizer.from_pretrained(qa_model_id)   # instant, cached by the setup cell
qa_model = DistilBertForQuestionAnswering.from_pretrained(qa_model_id)
qa_model.eval()
print("Extractive reader loaded.")

In [ ]:
def ask_question(question, context):
    inputs = qa_tokenizer(question, context, return_tensors="pt", truncation=True)
    with torch.no_grad():
        outputs = qa_model(**inputs)

    # softmax over all positions: how much probability mass lands on the chosen start/end?
    start_probs = F.softmax(outputs.start_logits, dim=-1)
    end_probs   = F.softmax(outputs.end_logits, dim=-1)
    start = torch.argmax(start_probs)
    end   = torch.argmax(end_probs)
    confidence = (start_probs[0, start] * end_probs[0, end]).item()

    if end < start:          # incoherent span: the closest this model gets to "no answer"
        return "", confidence
    answer = qa_tokenizer.decode(inputs.input_ids[0][start:end + 1])
    return answer, confidence

In [ ]:
# Our context is one real Helcon help-center article, the same documents the support copilot reads.
corpus = {d["slug"]: d for d in load_seed("corpus")}
shopify_article = corpus["connecting-your-store"]
CONTEXT = shopify_article["text"]

QUESTION = "Where do I manage integrations in Helcon?"
# try: QUESTION = "How do I connect Helcon to SAP?" (the answer is NOT in the article,
#      yet the model still produces a span; watch the confidence collapse).
# try: QUESTION = "Does Helcon support Shopify?" (a yes/no question; extractive QA cannot say "yes")

answer, confidence = ask_question(QUESTION, CONTEXT)
print(f"Article: {shopify_article['title']}")
print(f"Q: {QUESTION}")
print(f"A: '{answer}'   (confidence: {confidence:.1%})")

## Takeaway

The model has no "I don't know" button. Ask how to connect SAP, which the Shopify article never mentions, and it still points at some span in the text. Extraction is forced; what changes is the confidence, which collapses from a comfortable figure to a few percent. In production that probability is where you put your threshold: below it, escalate to a human or answer "not found". Yes/no questions are structurally impossible here, because the words yes and no are not in the article, so no span can contain them.

## Step 2: The 512-token wall

Our reader handled a single help-center article. A real support copilot does not get to pick the one right article in advance. It faces Helcon's whole knowledge base, dozens of articles and policies concatenated. BERT-family encoders have a hard limit: 512 tokens of question plus context, and no more. The next cell glues several Helcon articles together, buries the Shopify answer at the very end of the pile, and asks the same question. The function already passes `truncation=True`, so nothing crashes. Before running it, predict what the answer will be.

In [ ]:
# Helcon's help center, glued into one blob, with the Shopify article appended at the very END.
other_slugs = ["plans-and-pricing", "inviting-teammates", "understanding-your-invoice",
               "resetting-your-password", "importing-your-data"]
prefix = "\n\n".join(corpus[s]["text"] for s in other_slugs)
LONG_CONTEXT = prefix + "\n\n" + CONTEXT   # the only Shopify-answering article sits last

total_tokens = len(qa_tokenizer.encode(QUESTION, LONG_CONTEXT))
print(f"question + help center: {total_tokens:,} tokens | model limit: {qa_tokenizer.model_max_length}")

answer, confidence = ask_question(QUESTION, LONG_CONTEXT)   # truncation=True silently drops the end
print(f"A: '{answer}'   (confidence: {confidence:.1%})")

The tokenizer counted well over a thousand tokens; the model saw the first 512 and nothing else. `truncation=True` did not fix the problem, it hid it. The Shopify article sat at the end of the pile, past the window, so it was silently cut off, and the reader pointed at the most plausible span in the part it could see, confidence and all. The reader physically cannot see past its window.

This is why we chunk Helcon's knowledge base. The fix is not a bigger model, it is an architecture: cut every article into pieces, search the pieces for the few that might contain the answer, and hand only those to the reader. Retrieval plus reading is exactly the system we assemble over Helcon's knowledge base in the next session, and exactly what the letters of RAG stand for.

### Optional aside: pointing the reader at the open web

A tempting shortcut: instead of providing the context, google the question, scrape the first result, and feed it to the reader. The cell below does exactly that, and you should know why it never survives contact with production. Google actively blocks scrapers with captchas and IP rate limits, scraping violates the Terms of Service, and the HTML you parse today changes silently tomorrow. Real systems use a search API or their own private index. This cell is self-contained and optional; nothing later depends on it.

In [ ]:
# OPTIONAL AND FRAGILE: scraping Google works until it doesn't. Skip freely; nothing below needs it.
import requests
from bs4 import BeautifulSoup
from googlesearch import search

def fetch_relevant_webpages(query, num_results=1):
    return [url for url, _ in zip(search(query), range(num_results))]

def extract_text_from_webpage(url):
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")
    return " ".join(p.get_text() for p in soup.find_all("p"))

web_question = "Who won the last Soccer World Cup?"
try:
    url = fetch_relevant_webpages(web_question)[0]
    web_context = extract_text_from_webpage(url)
    answer, confidence = ask_question(web_question, web_context)   # the same 512-token truncation applies
    print(f"source: {url}")
    print(f"A: '{answer}'   (confidence: {confidence:.1%})")
except Exception as e:
    print(f"Scraping failed (expected, eventually): {e}")

## Step 3: A generative reader, T5

T5 treats every NLP task as text-to-text: feed it `question: ... context: ...` and it writes an answer token by token, with the same auto-regressive loop we saw in the language modeling lab. That buys two abilities extraction cannot offer, it can rephrase and it can say "yes", at a price you can already guess. A model that writes its answers can also write things that are not in the context.

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

T5_SIZE   = "t5-small"   # try: "t5-base", visibly better prose, noticeably slower on CPU
NUM_BEAMS = 1            # try: 8, beam search: fluency vs latency you can feel

t5_tokenizer = T5Tokenizer.from_pretrained(T5_SIZE, model_max_length=512)
t5_model = T5ForConditionalGeneration.from_pretrained(T5_SIZE)

def ask_question_generative(question, context):
    input_text = f"question: {question} context: {context}"
    inputs = t5_tokenizer(input_text, return_tensors="pt", truncation=True)
    outputs = t5_model.generate(inputs["input_ids"], max_length=64, num_beams=NUM_BEAMS)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Q: Where do I manage integrations in Helcon?")
print("A:", ask_question_generative("Where do I manage integrations in Helcon?", CONTEXT))
print()
print("Q: Does Helcon support Shopify?")   # the yes/no question extraction could not answer
print("A:", ask_question_generative("Does Helcon support Shopify?", CONTEXT))

## Takeaway

Two reader families, one trade. The extractive reader returns a span plus a probability: fast, free to run, and auditable, because the answer is literally in the text. It cannot abstain, cannot synthesize, and cannot say "yes". The generative reader composes fluent answers and handles yes/no questions, but nothing anchors its words to the context, so it can hallucinate with perfect grammar. Both are blind past their ~512-token window. The conclusion follows on its own: chunk the documents, retrieve the right chunks, and put a reader on top. That is the next session.